# 🧠 BrandPulse AI — Deep Learning Sentiment Classifier
### Notebook 02 · Bidirectional LSTM with TensorFlow / Keras

---

**Goal**: Train a sequence-aware **Bidirectional LSTM** that outperforms the classical TF-IDF baseline
by capturing word-order context and long-range dependencies within tweets.

| Stage | Details |
|---|---|
| Tokenisation | Keras `Tokenizer` · vocab = 10 k |
| Sequences | `pad_sequences` · maxlen = 100 · post-padding |
| Architecture | Embedding → SpatialDropout → BiLSTM → Dense |
| Training | Adam · `categorical_crossentropy` · EarlyStopping |
| Persistence | `models/lstm_model.keras` · `models/tokenizer.json` |

---
**Dataset**: Twitter US Airline Sentiment (Kaggle)  
**Author** : BrandPulse AI — Deep Learning Engineering Team  
**Kernel**  : Python 3.10+ · TensorFlow 2.16+ · Keras 3+

---
## 0 · Environment Setup & Imports

In [ ]:
# ── Standard library ──────────────────────────────────────────────
import os
import sys
import json
import time
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # suppress TF C++ info/warnings

# ── Numeric / Data ────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Machine Learning utilities ────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)

# ── TensorFlow / Keras ────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers
from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

# ── BrandPulse preprocessing ──────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.preprocessing import TextPreprocessor, preprocess_dataframe

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Global paths ──────────────────────────────────────────────────
DATA_PATH  = os.path.join(PROJECT_ROOT, 'data', 'Tweets.csv')
MODEL_DIR  = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Hyper-parameters (single source of truth) ────────────────────
VOCAB_SIZE   = 10_000
MAX_LEN      = 100
EMBED_DIM    = 128
LSTM_UNITS   = 64
DENSE_UNITS  = 32
NUM_CLASSES  = 3
BATCH_SIZE   = 64
EPOCHS       = 20
LR           = 0.001

# ── Global plot style (dark-mode BrandPulse theme) ────────────────
plt.rcParams.update({
    'figure.dpi'      : 120,
    'figure.facecolor': '#0F1117',
    'axes.facecolor'  : '#0F1117',
    'axes.edgecolor'  : '#2A2D3E',
    'axes.labelcolor' : '#E0E0E0',
    'axes.titlecolor' : '#FFFFFF',
    'text.color'      : '#E0E0E0',
    'xtick.color'     : '#9E9E9E',
    'ytick.color'     : '#9E9E9E',
    'grid.color'      : '#2A2D3E',
    'grid.linewidth'  : 0.5,
    'font.family'     : 'sans-serif',
    'font.size'       : 11,
})
LABEL_ORDER = ['negative', 'neutral', 'positive']
PALETTE     = {'negative': '#FF4E6A', 'neutral': '#FFC947', 'positive': '#4ECDC4'}

print('TensorFlow version :', tf.__version__)
print('Keras version      :', keras.__version__)
print('Python version     :', sys.version.split()[0])
gpus = tf.config.list_physical_devices('GPU')
print('GPU devices        :', gpus if gpus else 'None — running on CPU')

---
## 1 · Data Ingestion & Preprocessing

We reload `Tweets.csv` fresh and run the BrandPulse NLP pipeline to produce clean, lemmatised text.

> **Why re-run preprocessing here?** The LSTM consumes the cleaned text differently from TF-IDF — the Keras `Tokenizer` builds its own integer vocabulary from the same cleaned corpus, ensuring identical input semantics between the two notebooks.

In [ ]:
df_raw = pd.read_csv(DATA_PATH, encoding='utf-8')
df = df_raw[['airline_sentiment', 'text', 'airline']].copy()
df.dropna(subset=['text', 'airline_sentiment'], inplace=True)
df.rename(columns={'airline_sentiment': 'label'}, inplace=True)

print(f'Raw dataset : {len(df):,} rows')
print('Label counts:')
print(df['label'].value_counts().to_string())

In [ ]:
# Initialise preprocessor (spaCy primary, NLTK fallback)
tp = TextPreprocessor(spacy_model='en_core_web_sm', min_token_length=2)
print(f'NLP backend : {tp.backend}')

t0 = time.time()
df = preprocess_dataframe(df, text_col='text', out_col='clean_text', preprocessor=tp)
elapsed = time.time() - t0

# Drop empty post-clean rows
before = len(df)
df = df[df['clean_text'].str.strip().astype(bool)].reset_index(drop=True)
print(f'Preprocessing : {elapsed:.1f}s  |  dropped {before - len(df)} empty rows')
print(f'Usable rows   : {len(df):,}')
df[['text', 'clean_text', 'label']].sample(3, random_state=SEED)

---
## 2 · Label Encoding

We encode the three sentiment classes as **one-hot vectors** (required by `categorical_crossentropy`):

| String label | Integer | One-hot |
|---|---|---|
| `negative` | 0 | `[1, 0, 0]` |
| `neutral`  | 1 | `[0, 1, 0]` |
| `positive` | 2 | `[0, 0, 1]` |

The `LabelEncoder` also gives us a clean way to decode integer predictions back to strings at inference time.

In [ ]:
le = LabelEncoder()
le.fit(LABEL_ORDER)           # fix class order: neg=0, neu=1, pos=2
y_int   = le.transform(df['label'])          # integer labels
y_onehot = to_categorical(y_int, num_classes=NUM_CLASSES)  # one-hot

print('Label encoder classes :', le.classes_)
print('y_int  shape          :', y_int.shape)
print('y_onehot shape        :', y_onehot.shape)
print()
print('Example encodings:')
for i, label in enumerate(LABEL_ORDER):
    idx = np.where(le.transform([label]))[0]
    print(f'  {label:>10s}  →  int={le.transform([label])[0]}  →  one-hot={y_onehot[np.where(y_int == le.transform([label])[0])[0][0]]}')

---
## 3 · Stratified Train / Validation / Test Split

We create three partitions:

| Split | Size | Purpose |
|---|---|---|
| **Train** | 70 % | Model weight optimisation |
| **Validation** | 15 % | EarlyStopping + hyperparameter monitoring |
| **Test** | 15 % | Unbiased final evaluation (never seen during training) |

In [ ]:
X_text = df['clean_text'].values

# 70 / 30 first split
X_train_t, X_temp, y_train_t, y_temp, yi_train, yi_temp = train_test_split(
    X_text, y_onehot, y_int,
    test_size=0.30, random_state=SEED, stratify=y_int
)

# 15 val / 15 test from remaining 30%
X_val, X_test, y_val, y_test, yi_val, yi_test = train_test_split(
    X_temp, y_temp, yi_temp,
    test_size=0.50, random_state=SEED, stratify=yi_temp
)

print(f'Train      : {len(X_train_t):>5,}  ({len(X_train_t)/len(X_text):.0%})')
print(f'Validation : {len(X_val):>5,}  ({len(X_val)/len(X_text):.0%})')
print(f'Test       : {len(X_test):>5,}  ({len(X_test)/len(X_text):.0%})')
print()
print('Train label split (%):')
for cls, n in zip(le.classes_, np.bincount(yi_train)):
    print(f'  {cls:>10s}: {n:>4,}  ({n/len(yi_train)*100:.1f}%)')

---
## 4 · Keras Tokenization & Sequence Padding

The Keras `Tokenizer`:
- Builds a word-index vocabulary from the **training corpus only** (no data leakage).
- Maps out-of-vocabulary tokens to a reserved `<OOV>` index.
- Converts sentences to integer sequences that the Embedding layer can consume.

Post-padding (`padding='post'`) zeros are appended so the LSTM can still learn the actual final state from real content at the sequence end.

In [ ]:
# ── Fit Tokenizer on training corpus only ────────────────────────
tokenizer = Tokenizer(
    num_words=VOCAB_SIZE,
    oov_token='<OOV>',
    lower=True,
    filters='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n',
)
tokenizer.fit_on_texts(X_train_t)

vocab_size_actual = min(VOCAB_SIZE, len(tokenizer.word_index) + 1)
print(f'Unique tokens in training corpus : {len(tokenizer.word_index):,}')
print(f'Effective vocab size             : {vocab_size_actual:,}')
print()
print('Top 15 most frequent tokens:')
top15 = sorted(tokenizer.word_counts.items(), key=lambda x: -x[1])[:15]
for word, count in top15:
    print(f'  {word:<20s} {count:>5,}')

In [ ]:
# ── Text → integer sequences ──────────────────────────────────────
def texts_to_padded(texts, tokenizer, maxlen=MAX_LEN):
    """Convert a list of strings to a padded integer sequence array."""
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=maxlen, padding='post', truncating='post')

X_train_pad = texts_to_padded(X_train_t, tokenizer)
X_val_pad   = texts_to_padded(X_val,     tokenizer)
X_test_pad  = texts_to_padded(X_test,    tokenizer)

print(f'X_train_pad : {X_train_pad.shape}  dtype={X_train_pad.dtype}')
print(f'X_val_pad   : {X_val_pad.shape}')
print(f'X_test_pad  : {X_test_pad.shape}')

# ── Show a single example ─────────────────────────────────────────
idx = 0
print(f'\nExample tweet      : {X_train_t[idx]}')
print(f'Integer sequence   : {X_train_pad[idx][:20]} …')
print(f'One-hot label      : {y_train_t[idx]}  → {le.classes_[np.argmax(y_train_t[idx])]}')

In [ ]:
# ── Sequence length distribution ──────────────────────────────────
seq_lengths = np.array([len(tokenizer.texts_to_sequences([t])[0]) for t in X_train_t])

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(seq_lengths, bins=60, color='#7B61FF', edgecolor='#0F1117', linewidth=0.3, alpha=0.85)
ax.axvline(MAX_LEN, color='#FF4E6A', lw=2, linestyle='--', label=f'maxlen={MAX_LEN}')
ax.axvline(np.median(seq_lengths), color='#4ECDC4', lw=1.5, linestyle=':', label=f'median={np.median(seq_lengths):.0f}')
ax.set_title('Token Sequence Length Distribution (Training Set)', fontsize=12, fontweight='bold')
ax.set_xlabel('Sequence length (tokens)')
ax.set_ylabel('Count')
ax.legend(facecolor='#1E2030', edgecolor='#2A2D3E', labelcolor='white')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, '..', 'data', 'seq_length_dist.png'), bbox_inches='tight', facecolor='#0F1117')
plt.show()
print(f'Mean length  : {seq_lengths.mean():.1f} tokens')
print(f'Median length: {np.median(seq_lengths):.0f} tokens')
print(f'95th pct     : {np.percentile(seq_lengths, 95):.0f} tokens')
print(f'Truncated    : {(seq_lengths > MAX_LEN).sum()} / {len(seq_lengths)} sequences ({(seq_lengths > MAX_LEN).mean()*100:.1f}%)')

In [ ]:
# ── Persist tokenizer to JSON ─────────────────────────────────────
TOKENIZER_PATH = os.path.join(MODEL_DIR, 'tokenizer.json')
with open(TOKENIZER_PATH, 'w', encoding='utf-8') as f:
    f.write(tokenizer.to_json())

size_kb = os.path.getsize(TOKENIZER_PATH) / 1024
print(f'Tokenizer saved → {TOKENIZER_PATH}  ({size_kb:.1f} KB)')

# Reload check
with open(TOKENIZER_PATH, 'r', encoding='utf-8') as f:
    tok_reloaded = tokenizer_from_json(f.read())
print(f'Reload check   : vocab size = {len(tok_reloaded.word_index):,}  OK')

---
## 5 · Model Architecture

### Architecture Diagram

```
Input (100,)
     │
     ▼
Embedding(10000 → 128)          # Dense word vectors, trained end-to-end
     │
     ▼
SpatialDropout1D(0.2)           # Drop entire feature maps (better than scalar dropout for RNNs)
     │
     ▼
Bidirectional(LSTM(64))         # Forward + backward context  →  128-dim output
  dropout=0.2  recurrent_dropout=0.2
     │
     ▼
Dense(32, relu)                 # Classifier head
     │
     ▼
Dropout(0.3)
     │
     ▼
Dense(3, softmax)               # 3-class probability distribution
```

**Why Bidirectional?** Sentiment often depends on context from *both* sides of a word —
e.g., *"not bad"* or *"couldn't have been worse"* require reading past the keyword.

In [ ]:
def build_model(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    maxlen=MAX_LEN,
    lstm_units=LSTM_UNITS,
    dense_units=DENSE_UNITS,
    num_classes=NUM_CLASSES,
    lr=LR,
) -> keras.Model:
    """
    Build and compile the Bidirectional LSTM sentiment classifier.

    Returns
    -------
    keras.Model
        Compiled model ready for training.
    """
    model = keras.Sequential([
        # ── Input & Embedding ─────────────────────────────────────
        layers.Embedding(
            input_dim=vocab_size,
            output_dim=embed_dim,
            name='embedding',
        ),
        # ── Regularisation: drop full embedding channels ──────────
        layers.SpatialDropout1D(0.2, name='spatial_dropout'),
        # ── Bidirectional LSTM ────────────────────────────────────
        layers.Bidirectional(
            layers.LSTM(
                lstm_units,
                dropout=0.2,
                recurrent_dropout=0.2,
            ),
            name='bilstm',
        ),
        # ── Classifier head ───────────────────────────────────────
        layers.Dense(dense_units, activation='relu', name='dense_head'),
        layers.Dropout(0.3, name='head_dropout'),
        layers.Dense(num_classes, activation='softmax', name='output'),
    ], name='BrandPulse_BiLSTM')

    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
        ],
    )
    return model


model = build_model()
# Keras 3: input_length is deprecated — explicitly build so count_params()
# and model.summary() have complete shape information.
model.build(input_shape=(None, MAX_LEN))
model.summary()

In [ ]:
# ── Parameter count breakdown ─────────────────────────────────────
# Guard: ensure the model is built before calling count_params().
# model.build() was already called in the previous cell; this is a
# safety net in case cells are run out of order.
if not model.built:
    model.build(input_shape=(None, MAX_LEN))

total_params     = model.count_params()
trainable_params = sum(int(np.prod(v.shape)) for v in model.trainable_variables)

print(f'Total parameters     : {total_params:>12,}')
print(f'Trainable parameters : {trainable_params:>12,}')
print()
print('Layer-wise parameter breakdown:')
for layer in model.layers:
    lp = sum(int(np.prod(w.shape)) for w in layer.trainable_weights)
    if lp > 0:
        pct = lp / total_params * 100
        print(f'  {layer.name:<22s} {lp:>10,}  ({pct:.1f}%)')

---
## 6 · Training

### Callbacks

| Callback | Configuration | Purpose |
|---|---|---|
| `EarlyStopping` | `monitor='val_loss'`, `patience=3`, `restore_best_weights=True` | Halt when validation loss stops improving; revert to best checkpoint |
| `ReduceLROnPlateau` | `patience=2`, `factor=0.5`, `min_lr=1e-6` | Halve learning rate when stuck in a plateau |
| `ModelCheckpoint` | Save best `val_accuracy` model | Persist best weights during training |

> **Class weighting** is applied to compensate for the ~63 % negative-class majority without modifying the dataset.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Compute class weights from training integer labels
class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(yi_train),
    y=yi_train,
)
class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}

print('Class weights:')
for i, (cls, w) in enumerate(zip(le.classes_, class_weights_arr)):
    print(f'  {i} ({cls:>10s}) : {w:.4f}')

In [ ]:
CHECKPOINT_PATH = os.path.join(MODEL_DIR, 'best_bilstm_checkpoint.keras')

training_callbacks = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=0,
    ),
]

print('Callbacks configured:')
for cb in training_callbacks:
    print(f'  - {type(cb).__name__}')

In [ ]:
print('='*60)
print('  BrandPulse BiLSTM — Training')
print('='*60)
print(f'  Train      : {X_train_pad.shape[0]:,} samples')
print(f'  Validation : {X_val_pad.shape[0]:,} samples')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Max epochs : {EPOCHS}')
print('='*60)

t0 = time.time()
history = model.fit(
    X_train_pad, y_train_t,
    validation_data=(X_val_pad, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=training_callbacks,
    verbose=1,
)
elapsed = time.time() - t0
actual_epochs = len(history.history['loss'])
print(f'\nTraining complete: {actual_epochs} epochs in {elapsed:.1f}s  ({elapsed/actual_epochs:.1f}s / epoch)')

---
## 7 · Training Curves

Smooth convergence with a gap between train/val curves indicates mild overfitting — expected given the ~10 k training samples. `EarlyStopping` with `restore_best_weights` ensures we pick the sweet spot.

In [ ]:
hist = history.history
epochs_ran = range(1, len(hist['loss']) + 1)

fig = plt.figure(figsize=(15, 5))
fig.patch.set_facecolor('#0F1117')
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# ── (a) Loss ─────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0])
ax0.plot(epochs_ran, hist['loss'],     color='#7B61FF', lw=2,   label='Train loss')
ax0.plot(epochs_ran, hist['val_loss'], color='#FF4E6A', lw=2,   label='Val loss',   linestyle='--')
best_e = int(np.argmin(hist['val_loss'])) + 1
ax0.axvline(best_e, color='#FFC947', lw=1.2, linestyle=':', alpha=0.8, label=f'Best epoch={best_e}')
ax0.set_title('Loss', fontweight='bold')
ax0.set_xlabel('Epoch');  ax0.set_ylabel('Categorical Cross-entropy')
ax0.legend(facecolor='#1E2030', edgecolor='#2A2D3E', labelcolor='white', fontsize=9)
ax0.grid(alpha=0.3)

# ── (b) Accuracy ─────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[1])
ax1.plot(epochs_ran, hist['accuracy'],     color='#4ECDC4', lw=2, label='Train acc')
ax1.plot(epochs_ran, hist['val_accuracy'], color='#FFC947', lw=2, label='Val acc',   linestyle='--')
ax1.axvline(best_e, color='#FF4E6A', lw=1.2, linestyle=':', alpha=0.8)
ax1.set_title('Accuracy', fontweight='bold')
ax1.set_xlabel('Epoch');  ax1.set_ylabel('Accuracy')
ax1.legend(facecolor='#1E2030', edgecolor='#2A2D3E', labelcolor='white', fontsize=9)
ax1.grid(alpha=0.3)

# ── (c) Learning rate trace ───────────────────────────────────────
ax2 = fig.add_subplot(gs[2])
if 'lr' in hist:
    ax2.semilogy(epochs_ran, hist['lr'], color='#A78BFA', lw=2, marker='o', markersize=4)
    ax2.set_title('Learning Rate Schedule', fontweight='bold')
    ax2.set_xlabel('Epoch');  ax2.set_ylabel('LR (log scale)')
    ax2.grid(alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'LR history\nnot recorded', ha='center', va='center',
             transform=ax2.transAxes, color='#9E9E9E', fontsize=12)
    ax2.set_title('Learning Rate Schedule', fontweight='bold')

fig.suptitle('BrandPulse BiLSTM — Training Curves', fontsize=14, fontweight='bold', y=1.02)
plt.savefig(os.path.join(MODEL_DIR, '..', 'data', 'lstm_training_curves.png'), bbox_inches='tight', facecolor='#0F1117')
plt.show()

print(f'Best epoch (lowest val_loss)     : {best_e}')
print(f'Best val_loss                    : {min(hist["val_loss"]):.4f}')
print(f'Best val_accuracy                : {max(hist["val_accuracy"]):.4f}')

---
## 8 · Test Set Evaluation

We evaluate on the **held-out test split** (15 % — never seen during training or callback monitoring).

In [ ]:
# ── Raw metric evaluation ─────────────────────────────────────────
test_results = model.evaluate(X_test_pad, y_test, batch_size=BATCH_SIZE, verbose=0)
metric_names = ['Loss', 'Accuracy', 'Precision', 'Recall']

print('=== Test Set Keras Metrics ===')
for name, val in zip(metric_names, test_results):
    print(f'  {name:<12}: {val:.4f}')

In [ ]:
# ── sklearn Classification Report ────────────────────────────────
y_prob  = model.predict(X_test_pad, batch_size=BATCH_SIZE, verbose=0)
y_pred  = np.argmax(y_prob, axis=1)
y_true  = np.argmax(y_test, axis=1)

report = classification_report(
    y_true, y_pred,
    target_names=le.classes_,
    digits=4,
)
print('=== Classification Report (Test Set) ===')
print(report)

# per-class F1
f1_per_class = f1_score(y_true, y_pred, average=None, labels=[0, 1, 2])
macro_f1     = f1_score(y_true, y_pred, average='macro')
weighted_f1  = f1_score(y_true, y_pred, average='weighted')
accuracy     = accuracy_score(y_true, y_pred)

print(f'Macro F1    : {macro_f1:.4f}')
print(f'Weighted F1 : {weighted_f1:.4f}')
print(f'Accuracy    : {accuracy:.4f}')

In [ ]:
# ── Normalised Confusion Matrix ───────────────────────────────────
cm_norm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2], normalize='true')
cm_raw  = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0F1117')
fig.suptitle('BrandPulse BiLSTM — Confusion Matrices (Test Set)', fontsize=13, fontweight='bold', y=1.01)

for ax, cm, title, fmt in [
    (axes[0], cm_norm, 'Normalised (Row %)', '.2f'),
    (axes[1], cm_raw,  'Raw Counts',         'd'),
]:
    sns.heatmap(
        cm, annot=True, fmt=fmt,
        cmap='RdYlGn',
        xticklabels=le.classes_,
        yticklabels=le.classes_,
        linewidths=0.5, linecolor='#0F1117',
        vmin=(0 if fmt == '.2f' else None),
        vmax=(1 if fmt == '.2f' else None),
        ax=ax,
        cbar_kws={'shrink': 0.8},
    )
    ax.set_title(title, fontsize=11, fontweight='bold', color='white', pad=10)
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)
    ax.tick_params(colors='#E0E0E0')

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, '..', 'data', 'lstm_confusion_matrix.png'), bbox_inches='tight', facecolor='#0F1117')
plt.show()

In [ ]:
# ── Per-class F1 bar chart ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0F1117')

bar_colors = [PALETTE[l] for l in le.classes_]
bars = ax.bar(le.classes_, f1_per_class, color=bar_colors, edgecolor='#0F1117', linewidth=0.5, width=0.45)
ax.axhline(macro_f1,    color='#A78BFA', lw=1.8, linestyle='--', label=f'Macro F1 = {macro_f1:.4f}')
ax.axhline(weighted_f1, color='#FFC947', lw=1.8, linestyle=':',  label=f'Weighted F1 = {weighted_f1:.4f}')

for bar, val in zip(bars, f1_per_class):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10, color='white', fontweight='bold')

ax.set_ylim(0, 1.12)
ax.set_title('Per-Class F1 Score — BiLSTM (Test Set)', fontweight='bold', fontsize=12)
ax.set_ylabel('F1 Score')
ax.set_xlabel('Sentiment Class')
ax.legend(facecolor='#1E2030', edgecolor='#2A2D3E', labelcolor='white')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, '..', 'data', 'lstm_f1_by_class.png'), bbox_inches='tight', facecolor='#0F1117')
plt.show()

---
## 9 · Live Inference Tests

We test the model on three user-specified sample sentences plus additional airline-review edge cases.

In [ ]:
def predict_sentiment(texts, model, tokenizer, preprocessor, le, maxlen=MAX_LEN):
    """
    Run end-to-end inference on a list of raw tweet strings.

    Parameters
    ----------
    texts : list[str]
        Raw (uncleaned) tweet strings.
    model : keras.Model
        Trained BiLSTM model.
    tokenizer : Tokenizer
        Fitted Keras Tokenizer.
    preprocessor : TextPreprocessor
        BrandPulse NLP preprocessor instance.
    le : LabelEncoder
        Fitted sklearn LabelEncoder (int → string mapping).
    maxlen : int
        Sequence padding length.

    Returns
    -------
    list[dict]
        Each dict contains 'raw', 'cleaned', 'label', and per-class probabilities.
    """
    cleaned  = [preprocessor.clean(t) for t in texts]
    padded   = texts_to_padded(cleaned, tokenizer, maxlen=maxlen)
    probs    = model.predict(padded, verbose=0)
    preds    = np.argmax(probs, axis=1)
    labels   = le.inverse_transform(preds)

    results = []
    for raw, clean, label, prob in zip(texts, cleaned, labels, probs):
        results.append({
            'raw'     : raw,
            'cleaned' : clean,
            'label'   : label,
            'p_neg'   : round(float(prob[0]), 4),
            'p_neu'   : round(float(prob[1]), 4),
            'p_pos'   : round(float(prob[2]), 4),
        })
    return results


INFERENCE_SAMPLES = [
    # Required samples
    'The flight crew was exceptional and friendly!',
    'Waited 4 hours on the tarmac with no water. Terrible service.',
    # Additional edge cases
    '@AmericanAir my bag was lost AGAIN. This is completely unacceptable. #frustrated',
    'Flight was on time, nothing special but no complaints either.',
    '@SouthwestAir thank you for the quick rebooking — saved my trip!',
    'not bad for a budget airline I suppose',
]

preds = predict_sentiment(INFERENCE_SAMPLES, model, tokenizer, tp, le)

EMOJIS = {'positive': '😊', 'neutral': '😐', 'negative': '😡'}
COLORS_LABEL = {'positive': '\033[96m', 'neutral': '\033[93m', 'negative': '\033[91m'}
RESET = '\033[0m'

print('=' * 70)
print('  BrandPulse AI — Live Inference Results')
print('=' * 70)
for i, r in enumerate(preds, 1):
    emoji = EMOJIS[r['label']]
    print(f'[{i}] {emoji}  {r["label"].upper():>9}  (neg={r["p_neg"]:.3f} neu={r["p_neu"]:.3f} pos={r["p_pos"]:.3f})')
    print(f'     RAW    : {r["raw"]}')
    print(f'     CLEANED: {r["cleaned"]}')
    print()

In [ ]:
# ── Visualise inference confidence as stacked bars ────────────────
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#0F1117')

labels_short = [f'[{i+1}] {r["raw"][:35]}...' if len(r['raw']) > 35 else f'[{i+1}] {r["raw"]}'
                for i, r in enumerate(preds)]

p_neg = [r['p_neg'] for r in preds]
p_neu = [r['p_neu'] for r in preds]
p_pos = [r['p_pos'] for r in preds]

x   = np.arange(len(preds))
w   = 0.55
b1  = ax.bar(x, p_neg, w, label='Negative', color='#FF4E6A', edgecolor='#0F1117', lw=0.3)
b2  = ax.bar(x, p_neu, w, bottom=p_neg, label='Neutral', color='#FFC947', edgecolor='#0F1117', lw=0.3)
b3  = ax.bar(x, p_pos, w, bottom=np.array(p_neg)+np.array(p_neu), label='Positive', color='#4ECDC4', edgecolor='#0F1117', lw=0.3)

ax.set_xticks(x)
ax.set_xticklabels(labels_short, rotation=18, ha='right', fontsize=8)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Probability')
ax.set_title('BiLSTM Inference — Sentiment Probability Breakdown', fontweight='bold', fontsize=12)
ax.legend(facecolor='#1E2030', edgecolor='#2A2D3E', labelcolor='white', loc='upper right')
ax.grid(axis='y', alpha=0.25)
ax.axhline(0.5, color='white', lw=0.6, linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, '..', 'data', 'lstm_inference_probs.png'), bbox_inches='tight', facecolor='#0F1117')
plt.show()

---
## 10 · Model Persistence

The trained model is saved in the native **Keras format** (`.keras`), which bundles architecture, weights, and compile configuration in a single file — no need for separate weight files or JSON configs.

In [ ]:
MODEL_SAVE_PATH = os.path.join(MODEL_DIR, 'lstm_model.keras')
model.save(MODEL_SAVE_PATH)

size_mb = os.path.getsize(MODEL_SAVE_PATH) / 1024 / 1024
print(f'Model saved → {MODEL_SAVE_PATH}')
print(f'File size   : {size_mb:.2f} MB')

In [ ]:
# ── Reload & verify ───────────────────────────────────────────────
reloaded_model = keras.models.load_model(MODEL_SAVE_PATH)

# Re-run inference on the same samples to confirm output equality
y_prob_reload  = reloaded_model.predict(X_test_pad[:10], verbose=0)
y_prob_orig    = model.predict(X_test_pad[:10], verbose=0)
max_diff       = np.abs(y_prob_reload - y_prob_orig).max()

print('Reload verification:')
print(f'  Max probability diff (orig vs reloaded) : {max_diff:.2e}')
print(f'  Weights identical                        : {max_diff < 1e-6}')
print('\nAll artefacts saved successfully:')
for path in [MODEL_SAVE_PATH, TOKENIZER_PATH]:
    mb = os.path.getsize(path) / 1024 / 1024
    print(f'  {os.path.basename(path):<35s}  {mb:.2f} MB')

---
## 11 · Model Comparison: Classical vs. BiLSTM

We load the metrics from the classical baseline (if available) and render a direct head-to-head comparison.

In [ ]:
# ── Build comparison table ────────────────────────────────────────
bilstm_row = {
    'Model'       : 'BiLSTM (this notebook)',
    'Accuracy'    : round(accuracy, 4),
    'Macro F1'    : round(macro_f1, 4),
    'Weighted F1' : round(weighted_f1, 4),
    'F1 (neg)'    : round(f1_per_class[0], 4),
    'F1 (neu)'    : round(f1_per_class[1], 4),
    'F1 (pos)'    : round(f1_per_class[2], 4),
}

# Try to load classical pipeline and re-score on the same test split
CLASSICAL_PATH = os.path.join(MODEL_DIR, 'classical_tfidf_pipeline.joblib')
comparison_rows = [bilstm_row]

if os.path.exists(CLASSICAL_PATH):
    import joblib
    classical_pipe = joblib.load(CLASSICAL_PATH)
    # Classical pipeline expects cleaned text strings
    y_pred_cl = classical_pipe.predict(X_test)
    y_pred_cl_int = le.transform(y_pred_cl)

    f1_cl     = f1_score(yi_test, y_pred_cl_int, average=None, labels=[0, 1, 2])
    classic_row = {
        'Model'       : classical_pipe.named_steps['clf'].__class__.__name__,
        'Accuracy'    : round(accuracy_score(yi_test, y_pred_cl_int), 4),
        'Macro F1'    : round(f1_score(yi_test, y_pred_cl_int, average='macro'), 4),
        'Weighted F1' : round(f1_score(yi_test, y_pred_cl_int, average='weighted'), 4),
        'F1 (neg)'    : round(f1_cl[0], 4),
        'F1 (neu)'    : round(f1_cl[1], 4),
        'F1 (pos)'    : round(f1_cl[2], 4),
    }
    comparison_rows.append(classic_row)
    print('Classical pipeline loaded and scored.')
else:
    print('Classical pipeline not found — showing BiLSTM results only.')

cmp_df = pd.DataFrame(comparison_rows).set_index('Model')

def highlight_max(s):
    is_max = s == s.max()
    return ['background-color: #1D4E3F; color: #4ECDC4; font-weight: bold' if v else '' for v in is_max]

display(
    cmp_df.style
    .apply(highlight_max, axis=0)
    .format('{:.4f}')
    .set_caption('Model Showdown — Classical TF-IDF vs. BiLSTM')
)

---
## 12 · Summary & Next Steps

### Architecture Summary

```
Embedding(10k, 128) → SpatialDropout(0.2) → Bidirectional-LSTM(64)
  → Dense(32, relu) → Dropout(0.3) → Dense(3, softmax)
```

### Key Design Decisions

| Decision | Rationale |
|---|---|
| **Bidirectional LSTM** | Captures left-to-right *and* right-to-left context — crucial for negation ("not bad") |
| **SpatialDropout1D** | Drops entire embedding channels; more effective regularisation for sequence models than scalar dropout |
| **`recurrent_dropout`** | Applies dropout on the recurrent gate connections (not just outputs) — reduces overfitting in the LSTM core |
| **`class_weight='balanced'`** | Compensates for ~63 % negative-class majority without oversampling |
| **`EarlyStopping(restore_best_weights=True)`** | Prevents overfitting whilst automatically selecting the optimal checkpoint |
| **`ReduceLROnPlateau`** | Adaptively shrinks learning rate to escape loss plateaux |

### Saved Artefacts

| File | Location |
|---|---|
| Trained BiLSTM model | `models/lstm_model.keras` |
| Keras Tokenizer vocabulary | `models/tokenizer.json` |
| Classical TF-IDF pipeline | `models/classical_tfidf_pipeline.joblib` |

### Next Steps

> 🚀 **Notebook 03 / `app.py`** — Wire both models into the **Streamlit dashboard** with:
> - Real-time tweet sentiment analyser (single + batch CSV upload)
> - Side-by-side classical vs. LSTM prediction with confidence gauges
> - Interactive airline sentiment breakdown charts (Plotly)
> - WordCloud generator from the loaded dataset

---
*BrandPulse AI | Deep Learning Sentiment Classifier | Notebook 02*